In [ ]:
!pip -q install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time
spark = SparkSession.builder \
.appName("PySparkPractice") \
.getOrCreate()

In [ ]:
products_data = [
("P01", "Laptop", "Electronics", 60000),
("P02", "Monitor", "Electronics", 14000),
("P03", "Headphones", "Accessories", 3500),
("P04", "Printer", "Office", 17000),
("P05", "Router", "Networking", 4500),
("P06", "Tablet", "Electronics", 24000)
]
products_df = spark.createDataFrame(
products_data,
["product_id", "product_name", "category", "list_price"]
)
products_df.show()

In [ ]:
stores_data = [
("S01", "Hyderabad", "South"),
("S02", "Bengaluru", "South"),
("S03", "Mumbai", "West"),
("S04", "Delhi", "North")
]
stores_df = spark.createDataFrame(
stores_data, ["store_id", "city", "region"]
)
stores_df.show()

In [ ]:
sales_base_df = (
spark.range(1, 100001)
.withColumnRenamed("id", "order_id")
.withColumn(
"order_date",
F.date_add(
F.to_date(F.lit("2026-01-01")),
(F.col("order_id") % 180).cast("int")
)
)
.withColumn(
"product_id",
F.concat(
F.lit("P"),
F.lpad(
((F.col("order_id") % 6) + 1).cast("string"),
2, "0"
)
)
)
.withColumn(
"store_id",
F.concat(
F.lit("S"),
F.lpad(
((F.floor(F.col("order_id") / 3) % 4) + 1)
.cast("string"),
2, "0"
)
)
)
.withColumn(
"quantity",
((F.col("order_id") % 8) + 1).cast("int")
)
.withColumn(
"discount_pct",
F.when(
F.col("order_id") % 19 == 0,
F.lit(None).cast("int")
).otherwise(
((F.floor(F.col("order_id") / 5) % 4) * 5)
.cast("int")
)
)
)

In [ ]:
sales_df = (
sales_base_df
.withColumn(
"order_status",
F.when(F.col("order_id") % 11 == 0, "Cancelled")
.when(F.col("order_id") % 7 == 0, "Returned")
.otherwise("Completed")
)
.withColumn(
"payment_method",
F.when(F.col("order_id") % 3 == 0, "UPI")
.when(F.col("order_id") % 3 == 1, "Card")
.otherwise("Cash")
)
.withColumn(
"salesperson_id",
F.when(
F.col("order_id") % 17 == 0,
F.lit(None).cast("string")
).otherwise(
F.concat(
F.lit("EMP"),
F.lpad(
((F.col("order_id") % 12) + 1).cast("string"),
2, "0"
)
)
)
)
.join(
products_df.select(
"product_id",
F.col("list_price").alias("unit_price")
),
"product_id", "left"
)
.select(
"order_id", "order_date", "store_id",
"product_id", "quantity", "unit_price",
"discount_pct", "order_status",
"payment_method", "salesperson_id"
)
)
print("Total Records:", sales_df.count())
sales_df.printSchema()
sales_df.show(10, truncate=False)

Available DataFrames: sales_df (100,000 records), products_df (6 products), and stores_df (4
stores).

Part A | DataFrame transformations
1. Select order_id, product_id, quantity, and unit_price from the sales DataFrame.
2. Filter all completed transactions where quantity is greater than 5.
3. Create a column named gross_revenue by multiplying quantity and unit_price.
4. Rename order_status to transaction_status, then drop the payment_method column.

In [ ]:
#1
sales_df.select("product_id","quantity","unit_price").show(5)

In [ ]:
#2
from pyspark.sql.functions import col
sales_df.filter(col("quantity")>5).show(5)

In [ ]:
#3
sales_df=sales_df.withColumn("gross_revenue",col("quantity")*col("unit_price"))
sales_df.show(5)

In [ ]:
#4
sales_df=sales_df.withColumnRenamed("order_status","transaction_status").drop(col("payment_method"))
sales_df.show(5)

Part B | Aggregations and joins

5. Calculate the total quantity sold for each product.
6. Join sales with product master data and calculate total gross revenue by product category.
7. Join sales with store master data and calculate total transactions for each city.
8. Join all three DataFrames and find the top five products by gross revenue for completed orders.


In [ ]:
#5
join_df=(sales_df.join(products_df,on="product_id",how="inner")
.filter(col("transaction_status")=="Completed")
.groupBy(col("product_name"))
.agg(sum("quantity").alias("total_quantity"))
)
join_df.show()

In [ ]:
#6
join_df=(sales_df.join(products_df,on="product_id",how="inner")
.withColumn("gross_revenue",col("quantity")*col("unit_price"))
.groupBy(col("category"))
.agg(sum("gross_revenue").alias("total_gross_revenue")))
join_df.show()

In [ ]:
#7
from pyspark.sql.functions import count

join_df = (sales_df.join(stores_df, on="store_id", how="inner")
           .groupBy(col("city"))
           .agg(count("*").alias("total_transactions")))

join_df.show()

In [ ]:
#8
join_df=(sales_df.join(products_df,on="product_id",how="inner")
.join(stores_df,on="store_id",how="inner")
.filter(col("transaction_status")=="Completed")
.withColumn("gross_revenue",col("quantity")*col("unit_price"))
.groupBy(col("product_name"))
.agg(sum("gross_revenue").alias("gross_revenue"))
.orderBy(col("gross_revenue").desc())
)
join_df.show(5)


Part C | Null handling and conditions

9. Count null values in discount_pct and salesperson_id.
10. Replace null discounts with zero using na.fill().
11. Remove records where salesperson_id is null using na.drop().
12. Create order_size using when() and otherwise(): Small for quantity 1-3, Medium for 4-6, and Large
for 7 or more.

In [ ]:
#9
from pyspark.sql.functions import when
count_df=sales_df.agg(count(when(col("discount_pct").isNull(),1)).alias("discount_nulls_count"),
count(when(col("salesperson_id").isNull(),1)).alias("salesperson_id nulls"))

count_df.show()


In [ ]:
#10
sales_df=sales_df.na.fill({"discount_pct":0})
sales_df.select("discount_pct").show(5)

In [ ]:
#11
sales_df=sales_df.na.drop(subset=["salesperson_id"])
sales_df.show(5)

In [ ]:
#12
sales_df=sales_df.withColumn("order_size",
  when(col("quantity")<=3,"Small")
  .when(col("quantity")<=6,"Medium")
  .otherwise("Large")
)
sales_df.select("quantity","order_size").show(5)

Part D | Window and date functions

13. Rank products by total gross revenue within each category using dense_rank().
14. Calculate running gross revenue by store, ordered by date and order ID.
15. Extract the month from order_date and calculate monthly sales revenue for completed orders.

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col,
    row_number,
    rank,
    dense_rank,
    sum,
    avg,
    lag,
    lead
)

In [ ]:
#13
df1=(sales_df.join(products_df,on="product_id",how="inner")
.groupBy("category","product_name")
.agg(sum("gross_revenue").alias("total_gross_revenue")))
window_spec=Window.partitionBy("category").orderBy(col("total_gross_revenue").desc())
rank_df=df1.withColumn("revenue_rank",dense_rank().over(window_spec))
rank_df.orderBy("category", "revenue_rank").show()

In [ ]:
#15
from pyspark.sql.functions import sum
from pyspark.sql.functions import  month
df2=(sales_df.filter(col("transaction_status")=="Completed")
.withColumn("Month",month("order_date"))
.groupBy(col("Month"))
.agg(sum("gross_revenue").alias("monthly_revenue")))
df2.orderBy(col("Month")).show()



Part E | Lazy execution

16. Create three consecutive transformations: filter(), withColumn(), and select(). Add a Python
print() after each transformation. Trigger execution only at the end using show().
17. Create an aggregation by product_id. Use explain() to inspect the execution plan before calling
collect().

In [ ]:
from pyspark.sql.functions import lit
#16
df1=sales_df.filter(col("transaction_status")=="Completed")
print("Transformation 1 done!")
df2=sales_df.withColumn("Country",lit("India"))
print("Transformation 2 done!")
df3=df2.select("country","unit_price")
print("Transformation 3 done!")
df1.show(2)
df2.show(2)
df3.show(2)

In [ ]:
#17
agg_df = (sales_df
          .groupBy("product_id")
          .agg(count("*").alias("total_orders"),
          sum("gross_revenue").alias("total_revenue")))

agg_df.explain()

result = agg_df.collect()
print(result)

Part F | Caching and persistence

18. Filter completed orders, call cache(), and execute count() twice. Measure the execution time of
each action.
19. Persist transactions with quantity greater than 5 using StorageLevel.MEMORY_AND_DISK. Execute
two actions, then release the persisted data using unpersist().
20. Cache an aggregated DataFrame, inspect its execution plan, and identify InMemoryTableScan or
InMemoryRelation.

In [ ]:
#18
df=sales_df.filter(col("transaction_status")=="Completed")
df.cache()
print("Is cached: ",df.is_cached)
#first execution
start=time.perf_counter()
df.count()
time1=time.perf_counter()-start
#second execution
start=time.perf_counter()
df.count()
time2=time.perf_counter()-start
print("First Execution:", round(time1, 3), "seconds")
print("Second Execution:", round(time2, 3), "seconds")


In [ ]:
#19
#Create filtered DataFrame
high_quantity_df=sales_df.filter(F.col("quantity")>5)
#Persist using Memory and DISK
high_quantity_df.persist(StorageLevel.MEMORY_AND_DISK)
#ACTION 1
print("Count: ",high_quantity_df.count())
#ACTION 2
high_quantity_df.show(10)
#before unpersist
print("before unpersist", high_quantity_df.is_cached)
#Release the persisted data
high_quantity_df.unpersist()
print("after unpersist: ", high_quantity_df.is_cached)




In [ ]:
#20
agg_df=(sales_df.filter(col("transaction_status")=="Completed")
.groupBy("product_id")
.agg(sum("gross_revenue").alias("total_revenue"),
     count("*").alias("total_orders")))
agg_df.cache()
print("Plan BEFORE the cache is filled:")
agg_df.explain()

agg_df.count()   # first action fills the cache
print("Plan AFTER the cache is filled:")
agg_df.explain()


Part G | Repartition and coalesce

21. Check the existing partition count, then repartition the sales DataFrame into 8 partitions.
22. Display the number of records in each partition using spark_partition_id().
23. Repartition the sales DataFrame into 6 partitions based on store_id. Inspect which stores appear
in each partition.
24. Create a DataFrame with 12 partitions, then reduce it to 4 using coalesce(). Compare its
execution plan with repartition(4).
25. Save the sales data into 4 CSV part files using repartition(4) and write.csv().

In [ ]:
#21
partitioned_df=sales_df.repartition(8)
print("Original Partitions: ",sales_df.rdd.getNumPartitions())
print("New Partitions: ",partitioned_df.rdd.getNumPartitions())


In [ ]:
#22
from pyspark.sql import functions as F

distribution_df = partitioned_df.withColumn("partition_id", F.spark_partition_id())

distribution_df.groupBy("partition_id").count().orderBy("partition_id").show()

In [ ]:
#23
from pyspark.sql import functions as F

repartition_df = sales_df.repartition(6, "store_id")   # step 1: repartition

(repartition_df                                         # step 2: inspect
    .withColumn("partition_id", F.spark_partition_id())
    .groupBy("partition_id")
    .agg(F.collect_set("store_id").alias("stores"))
    .orderBy("partition_id")
    .show(truncate=False))

In [ ]:
#24
df_12 = sales_df.repartition(12)
print("Start:", df_12.rdd.getNumPartitions())

coalesce_df = df_12.coalesce(4)
repartition_df = df_12.repartition(4)

print("coalesce:", coalesce_df.rdd.getNumPartitions())
print("repartition:", repartition_df.rdd.getNumPartitions())

print("---- coalesce plan ----")
coalesce_df.explain()

print("---- repartition plan ----")
repartition_df.explain()

In [ ]:
#25
(sales_df
    .repartition(4)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv("output/sales_csv"))

Final Challenge

End-to-end PySpark data-processing pipeline
Build a single pipeline using sales_df, products_df, and stores_df. Complete the following steps:
1. Filter completed orders.
2. Replace null discounts with zero.
3. Join product and store details.
4. Calculate gross revenue and discounted revenue.
5. Group the data by region and product category.
6. Calculate total revenue and transaction count.
7. Rank categories by revenue within each region.
8. Cache the final aggregated DataFrame.
9. Display the results.
10. Write the output into 2 Parquet part files.

In [ ]:
#1
sales_df.filter(col("transaction_status")=="Completed").show(5)

In [ ]:
#2
sales_df.na.fill({"discount_pct":0}).show(5)

In [ ]:
#3
join_df = (sales_df
           .join(products_df, on="product_id", how="inner")
           .join(stores_df, on="store_id", how="inner"))

join_df.show(5)

In [ ]:
df1=sales_df.withColumn("gross_revenue",col("quantity")*col("unit_price"))
df2=sales_df.withColumn("discount_revenue",col("gross_revenue")*(1-col("discount_pct")/100))
df1.select("order_id", "quantity", "unit_price","gross_revenue").show(5)
df2.select("order_id", "quantity", "unit_price","gross_revenue","discount_revenue").show(5)

In [ ]:
#5
df = (sales_df
      .join(products_df, on="product_id", how="inner")
      .join(stores_df, on="store_id", how="inner")
      .groupBy("region", "category")
      .agg(count("*").alias("total_orders")))

df.show()

In [ ]:
#6
df=sales_df.agg(sum("gross_revenue").alias("total_revenue"),
                count("*").alias("Transaction_count"))
df.show(5)

In [ ]:
#7

region_cat_df = (sales_df
    .join(products_df, on="product_id", how="inner")
    .join(stores_df, on="store_id", how="inner")
    .groupBy("region", "category")
    .agg(sum("gross_revenue").alias("total_revenue")))

window_spec = Window.partitionBy("region").orderBy(col("total_revenue").desc())

final_df = region_cat_df.withColumn("revenue_rank", dense_rank().over(window_spec))

In [ ]:
#8
final_df.cache()

In [ ]:
#9
final_df.orderBy("region", "revenue_rank").show()

In [ ]:

#10
(final_df
    .repartition(2)
    .write
    .mode("overwrite")
    .parquet("output/region_category_parquet"))